In [0]:
import pandas as pd

In [0]:
spark.sql("use catalog proyecto_final_prueba")

In [0]:
catalog = spark.sql(
    "select current_catalog()"
).first()[0]

schema = "gold"
table = "fact_weather"

In [0]:
spark.sql(
    f"create schema if not exists {catalog}.{schema}"
)

spark.sql(
    f"drop table if exists {catalog}.{schema}.{table}"
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table} (
    id_tiempo BIGINT,
    id_ubicacion BIGINT,
    id_clima BIGINT,
    temperature DOUBLE,
    humidity INT,
    wind_speed DOUBLE
)
""")

In [0]:
df_silver = spark.table(
    f"{catalog}.silver.weather"
).toPandas()

df_silver

In [0]:
dim_tiempo = spark.table(
    f"{catalog}.gold.dim_tiempo"
).toPandas()

dim_ubicacion = spark.table(
    f"{catalog}.gold.dim_ubicacion"
).toPandas()

dim_clima = spark.table(
    f"{catalog}.gold.dim_clima"
).toPandas()

In [0]:
#relacion dim_tiempo
df_fact = df_silver.merge(
    dim_tiempo[
        [
            "id_tiempo",
            "fecha_hora_local"
        ]
    ],
    on="fecha_hora_local",
    how="left"
)

df_fact

In [0]:
#relacion dim_ubicacion
df_fact = df_fact.merge(
    dim_ubicacion[
        [
            "id_ubicacion",
            "latitude",
            "longitude"
        ]
    ],
    on=[
        "latitude",
        "longitude"
    ],
    how="left"
)

df_fact

In [0]:
#relacion dim_clima
df_fact = df_fact.merge(
    dim_clima[
        [
            "id_clima",
            "weather_code"
        ]
    ],
    on="weather_code",
    how="left"
)

df_fact

In [0]:
#seleccionamos columnas fact
columnas_fact = [
    "id_tiempo",
    "id_ubicacion",
    "id_clima",
    "temperature",
    "humidity",
    "wind_speed"
]

fact = df_fact[columnas_fact]

fact

In [0]:
#llaves
print("Registros totales:", len(fact))

print(
    "Registros sin id_tiempo:",
    fact["id_tiempo"].isna().sum()
)

print(
    "Registros sin id_ubicacion:",
    fact["id_ubicacion"].isna().sum()
)

print(
    "Registros sin id_clima:",
    fact["id_clima"].isna().sum()
)

In [0]:
fact = fact.drop_duplicates(
    subset=[
        "id_tiempo",
        "id_ubicacion"
    ]
)

In [0]:
df_spark = spark.createDataFrame(fact)

df_spark.display()

df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{catalog}.{schema}.{table}"
    )